In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count

# Create Spark session
spark = SparkSession.builder.appName("DuplicateCheck").getOrCreate()

# Dummy Data
data = [
    (1, "A", 100),
    (1, "A", 100),
    (2, "B", 200),
    (3, "C", 300),
    (2, "B", 200),
    (4, "D", 400)
]

columns = ["id", "category", "amount"]

df = spark.createDataFrame(data, columns)

# Identify duplicates using composite key (id, category)
duplicate_df = (
    df.groupBy("id", "category")
      .agg(count("*").alias("cnt"))
      .filter(col("cnt") > 1)
)

duplicate_df.show()

# Or

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("id", "category").orderBy(col("amount"))

df_with_row_num = df.withColumn("row_num", row_number().over(window_spec))

duplicates = df_with_row_num.filter(col("row_num") > 1)

duplicates.show()